# **Dataset Generator — Sketch → Cartoon & Live Action → Cartoon**

This notebook handles the construction of the two datasets used to train the style transfer GAN models. It covers two independent pipelines depending on the source domain.

---

## **Pipeline 1 — Live Action → Cartoon**

This pipeline processes `.mp4` video files of either live action footage or Disney cartoon scenes and extracts individual face images ready for training.

### How it works

1. **Frame extraction** — the video is read frame by frame, but not every frame is used. A `frame_skip` parameter controls how many frames are skipped between each sample, avoiding redundant near-identical images.
2. **Face detection** — each sampled frame is passed through a YOLOv8 model (`yolov8n.pt`) that detects whether a person (face) is present. Frames with no detection are skipped.
3. **Face alignment** — the detected bounding box is used to crop the face region and center it on a square canvas, which is then resized to `512x512`. This ensures all images in the dataset have a consistent size and face position.
4. **Landmark & mask generation** — approximate facial landmarks (eyes, nose, mouth corners) are estimated from the bounding box proportions, and a background mask is generated using `rembg`. Both are saved alongside the image for potential use in other tasks.
5. **Output** — images, masks, and landmark `.txt` files are saved to separate folders with sequential numeric filenames.

### Output structure
```
dataset_GAN_faces_aligned/
└── data/
    ├── train_cartoon/    ← aligned face images (512x512)
    ├── landmark/         ← .txt files with 5 approximate landmarks
    └── mask/             ← binary background masks
```


In [ ]:
!pip install insightface onnxruntime-gpu rembg opencv-python tqdm ultralytics

In [ ]:
import os
import cv2
import glob
import random
import shutil
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from rembg import remove


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.makedirs("/content/dataset_GAN_faces_aligned", exist_ok=True)

In [ ]:
# Cargar modelo YOLOv8
yolo_model = YOLO("yolov8n.pt")  # modelo pequeño, rápido, puedes cambiar a yolov8s.pt para más precisión

In [ ]:
# Helper Functions
def detect_face_yolo(frame, conf=0.3):
    """Detect if there's a face with YOLO"""
    results = yolo_model.predict(frame, imgsz=640, conf=conf)

    if len(results) == 0 or len(results[0].boxes) == 0:
        return None

    # Take the first assumed face
    box = results[0].boxes[0].xyxy[0].cpu().numpy()  # xmin, ymin, xmax, ymax
    xmin, ymin, xmax, ymax = map(int, box)
    return (xmin, ymin, xmax, ymax)

def face_align_512_yolo(frame, box):
    """Align face to 512x512 size"""
    imgSize = (512, 512)
    xmin, ymin, xmax, ymax = box
    face = frame[ymin:ymax, xmin:xmax]

    # Center in squared canvas
    h, w = face.shape[:2]
    dim = max(h, w)
    top = (dim - h) // 2
    left = (dim - w) // 2

    canvas = 255 * np.ones((dim, dim, 3), dtype=np.uint8)
    canvas[top:top+h, left:left+w] = face

    aligned = cv2.resize(canvas, imgSize, interpolation=cv2.INTER_AREA)
    return aligned

In [ ]:
def create_mask(img):
    """Create mask"""
    result = remove(img)
    mask = (result[:, :, 3] > 0).astype(np.uint8) * 255
    return mask

def get_landmarks_from_box(box):
    """Generate 5 landmarks using YOLO's box"""
    xmin, ymin, xmax, ymax = box
    w = xmax - xmin
    h = ymax - ymin

    landmarks = np.array([
        [xmin + 0.3*w, ymin + 0.3*h],  # left eye
        [xmin + 0.7*w, ymin + 0.3*h],  # right eye
        [xmin + 0.5*w, ymin + 0.55*h], # nose
        [xmin + 0.35*w, ymin + 0.75*h],# mouth left
        [xmin + 0.65*w, ymin + 0.75*h] # mouth right
    ], dtype=np.float32)

    return landmarks

In [ ]:
def process_videos(video_dir, out_img, out_lm, out_mask, frame_skip=8):
    """Process all videos on a file path"""
    os.makedirs(out_img, exist_ok=True)
    os.makedirs(out_lm, exist_ok=True)
    os.makedirs(out_mask, exist_ok=True)

    videos = [v for v in os.listdir(video_dir) if v.endswith(".mp4")]
    img_id = 0
    total_saved = 0
    total_skipped = 0

    for video in videos:
        cap = cv2.VideoCapture(os.path.join(video_dir, video))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        pbar = tqdm(range(total), desc=f"Processing {video}", leave=True)
        for i in pbar:
            ret, frame = cap.read()
            if not ret:
                break

            if i % frame_skip != 0:
                continue

            box = detect_face_yolo(frame)
            if box is None:
                total_skipped += 1
                continue

            # Approx landmarks
            landmarks = get_landmarks_from_box(box)

            aligned = face_align_512_yolo(frame, box)
            mask = create_mask(aligned)

            # Save results
            name = f"{img_id:06d}.png"
            cv2.imwrite(os.path.join(out_img, name), aligned)
            cv2.imwrite(os.path.join(out_mask, name), mask)
            # Save landmarks
            with open(os.path.join(out_lm, f"{img_id:06d}.txt"), "w") as f:
                for x, y in landmarks:
                    f.write(f"{int(x)} {int(y)}\n")

            img_id += 1
            total_saved += 1

            pbar.set_postfix({
                "saved": total_saved,
                "skipped": total_skipped
            })

        cap.release()

    print("\n----- DONE -----")
    print("Total saved:", total_saved)
    print("Total skipped:", total_skipped)

In [ ]:
process_videos(
    "/content/drive/MyDrive/GAN/cartoon_faces",
    "/content/dataset_GAN_faces_aligned/data/train_cartoon",
    "/content/dataset_GAN_faces_aligned/data/landmark",
    "/content/dataset_GAN_faces_aligned/data/mask",
    frame_skip=6
)

Processing PP2.mp4:   0%|          | 0/2565 [00:00<?, ?it/s]


0: 384x640 1 kite, 9.3ms
Speed: 2.4ms preprocess, 9.3ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)


Processing PP2.mp4:   0%|          | 1/2565 [00:00<37:39,  1.13it/s, saved=1, skipped=0]


0: 384x640 1 kite, 7.0ms
Speed: 3.3ms preprocess, 7.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)


Processing PP2.mp4:   0%|          | 7/2565 [00:01<09:13,  4.62it/s, saved=2, skipped=0]

...


0: 384x640 1 person, 10.1ms
Speed: 3.0ms preprocess, 10.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)


Processing pp3.mp4: 100%|█████████▉| 2011/2021 [02:49<00:01,  8.15it/s, saved=557, skipped=500]


0: 384x640 1 person, 9.6ms
Speed: 2.9ms preprocess, 9.6ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)


Processing pp3.mp4: 100%|██████████| 2021/2021 [02:49<00:00, 11.89it/s, saved=558, skipped=500]


----- DONE -----
Total saved: 558
Total skipped: 500


In [ ]:
def cartoon_to_pencil(img, blur_ksize=(5,5), edge_th=100):
    """
    Convierte imagen estilo cartoon/animada a lápiz.

    Args:
        img: imagen BGR
        blur_ksize: suavizado antes de extraer bordes
        edge_th: threshold para Canny

    Returns:
        sketch: imagen estilo lápiz
    """
    # Make grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Smooth
    gray_blur = cv2.GaussianBlur(gray, blur_ksize, 0)

    # Detect borders
    edges = cv2.Canny(gray_blur, threshold1=edge_th//2, threshold2=edge_th)

    # Invert color right back
    sketch = 255 - edges

    # Apply a light blur (pencil type)
    sketch = cv2.GaussianBlur(sketch, (3,3), 0)

    return sketch

In [ ]:
input_folder = "/content/dataset_GAN_faces_aligned/data/train"
output_folder = "/content/dataset_GAN_faces_aligned/sketch/train"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith((".png", ".jpg"))]

for filename in tqdm(files):

    img_path = os.path.join(input_folder, filename)
    img = cv2.imread(img_path)
    sketch = cartoon_to_pencil(img)

    cv2.imwrite(os.path.join(output_folder, filename), sketch)

print("Sketches generados")

100%|██████████| 135/135 [00:01<00:00, 104.20it/s]

Sketches generados


In [ ]:
import shutil

shutil.rmtree("/content/dataset_GAN_faces_aligned/data/train", ignore_errors=True)

In [ ]:
input_folder = "/content/dataset_GAN_faces_aligned/data/train_cartoon"
output_folder = "/content/dataset_GAN_faces_aligned/data/train_sketch"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".jpg") or filename.endswith(".png"):

        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)

        # Resizing
        img = cv2.resize(img, (512, 512))

        # Grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Pencil Sketch
        inv = 255 - gray
        blur = cv2.GaussianBlur(inv, (21, 21), 0)
        sketch = cv2.divide(gray, 255 - blur, scale=256)

        # Borders (Canny)
        edges = cv2.Canny(gray, 50, 150)

        # Invert edges
        edges = 255 - edges

        # Combine
        combined = cv2.addWeighted(sketch, 0.7, edges, 0.3, 0)

        # Turn up contrast
        combined = cv2.normalize(combined, None, 0, 255, cv2.NORM_MINMAX)

        # Umbral (darker lines)
        _, combined = cv2.threshold(combined, 200, 255, cv2.THRESH_BINARY)

        # Save
        output_path = os.path.join(output_folder, filename)
        cv2.imwrite(output_path, combined)

print("Sketches creados yupi")

Sketches creados yupi


In [ ]:
def create_image_pairs(fold_A, fold_B, fold_AB, num_imgs=1000000, use_AB=False):
    """
    Combines images from two folders (A and B) side by side into a third folder (AB).
    Compatible with flat directory structures (no subfolders).

    Args:
        fold_A (str): path to folder containing images A
        fold_B (str): path to folder containing images B
        fold_AB (str): output folder path
        num_imgs (int): maximum number of images to process
        use_AB (bool): if True, only combine files with '_A' and '_B' in their names
    """
    os.makedirs(fold_AB, exist_ok=True)

    img_list = os.listdir(fold_A)
    if use_AB:
        img_list = [img_path for img_path in img_list if '_A.' in img_path]

    num_to_use = min(num_imgs, len(img_list))
    print(f'Usando {num_to_use}/{len(img_list)} imágenes')

    for n in range(num_to_use):
        name_A = img_list[n]
        path_A = os.path.join(fold_A, name_A)

        if use_AB:
            name_B = name_A.replace('_A.', '_B.')
            name_AB = name_A.replace('_A.', '.')
        else:
            name_B = name_A
            name_AB = name_A

        path_B = os.path.join(fold_B, name_B)
        path_AB = os.path.join(fold_AB, name_AB)

        if os.path.isfile(path_A) and os.path.isfile(path_B):
            im_A = cv2.imread(path_A, cv2.IMREAD_COLOR)
            im_B = cv2.imread(path_B, cv2.IMREAD_COLOR)
            im_AB = np.concatenate([im_A, im_B], axis=1)
            cv2.imwrite(path_AB, im_AB)

# EJEMPLO
create_image_pairs(
    fold_A="/content/dataset_GAN_faces_aligned/data/train_sketch",
    fold_B="/content/dataset_GAN_faces_aligned/data/train_cartoon",
    fold_AB="/content/dataset_GAN_faces_aligned/data/train",
    num_imgs=558,
    use_AB=False
)

Usando 558/558 imágenes


In [ ]:
!zip -r /content/drive/MyDrive/GAN/dataset_GAN_faces_aligned.zip /content/dataset_GAN_faces_aligned/data

---

## **Pipeline 2 — Smooth Image Generation**

AnimeGAN's training requires a "smoothed" version of the cartoon images to compute the grayscale style loss. Without this the discriminator cannot learn to penalize overly harsh textures in the generator output.

### How it works

Each cartoon image goes through two operations:
1. **Morphological erosion** with an elliptical kernel — slightly shrinks bright regions and softens edges.
2. **Gaussian blur** — further smooths the result to reduce high-frequency texture detail.

The output images are saved to a `smooth/` folder inside the cartoon directory, which is exactly where the AnimeGAN training script looks for them automatically.

### Output structure
```
dataset_gan/train/cartoon/
├── style/     ← original cartoon frames
└── smooth/    ← smoothed versions for AnimeGAN training
```


In [ ]:
# Helper Yolo functions
def detect_face_yolo(frame, conf=0.3):
    """Detect faces in frame using YOLO"""
    results = yolo_model.predict(frame, imgsz=640, conf=conf)

    if len(results) == 0 or len(results[0].boxes) == 0:
        return None

    box = results[0].boxes[0].xyxy[0].cpu().numpy()
    xmin, ymin, xmax, ymax = map(int, box)
    return (xmin, ymin, xmax, ymax)

def face_align_256_yolo(frame, box):
    xmin, ymin, xmax, ymax = box
    h_img, w_img = frame.shape[:2]

    w = xmax - xmin
    h = ymax - ymin
    dim = max(w, h)

    cx = (xmin + xmax) // 2
    cy = (ymin + ymax) // 2

    x1 = cx - dim // 2
    y1 = cy - dim // 2
    x2 = x1 + dim
    y2 = y1 + dim

    # If it's offset eliminate
    if x1 < 0 or y1 < 0 or x2 > w_img or y2 > h_img:
        return None

    face = frame[y1:y2, x1:x2]
    aligned = cv2.resize(face, (256, 256), interpolation=cv2.INTER_AREA)
    return aligned

In [ ]:
def process_videos(video_dir, out_img, frame_skip=8):
    """Process videos"""
    os.makedirs(out_img, exist_ok=True)

    videos = [v for v in os.listdir(video_dir) if v.endswith(".mp4")]
    img_id = 0
    total_saved = 0
    total_skipped = 0

    for video in videos:
        cap = cv2.VideoCapture(os.path.join(video_dir, video))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        pbar = tqdm(range(total), desc=f"Processing {video}", leave=True)
        for i in pbar:
            ret, frame = cap.read()
            if not ret:
                break

            if i % frame_skip != 0:
                continue

            box = detect_face_yolo(frame)
            if box is None:
                total_skipped += 1
                continue

            aligned = face_align_256_yolo(frame, box)

            if aligned is None:
                total_skipped += 1
                continue

            cv2.imwrite(os.path.join(out_img, f"{img_id:06d}.png"), aligned)
            img_id += 1
            total_saved += 1

            pbar.set_postfix({
                "saved": total_saved,
                "skipped": total_skipped
            })

        cap.release()

    print("\n----- DONE -----")
    print("Total saved:", total_saved)
    print("Total skipped:", total_skipped)


In [ ]:
# LA Domain
process_videos(
    video_dir="/content/drive/MyDrive/GAN/live_action",
    out_img="/content/dataset_gan/train/live_action",
    frame_skip=10
)

# Cartoon Domain
process_videos(
    video_dir="/content/drive/MyDrive/GAN/cartoon",
    out_img="/content/dataset_gan/train/cartoon/style",
    frame_skip=8
)

Processing Aurora LA2.mp4:   0%|          | 0/5647 [00:00<?, ?it/s]


0: 384x640 1 person, 8.4ms
Speed: 3.4ms preprocess, 8.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)


Processing Aurora LA2.mp4:   0%|          | 5/5647 [00:00<01:55, 48.71it/s]


0: 384x640 1 person, 13.4ms
Speed: 7.0ms preprocess, 13.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)


Processing Aurora LA2.mp4:   0%|          | 19/5647 [00:00<01:36, 58.05it/s]

...


0: 384x640 8 persons, 8.7ms
Speed: 3.7ms preprocess, 8.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)


Processing Aurora Car.mp4: 100%|█████████▉| 8392/8397 [02:33<00:00, 49.83it/s, saved=474, skipped=575]


0: 384x640 6 persons, 1 kite, 8.8ms
Speed: 3.4ms preprocess, 8.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)


Processing Aurora Car.mp4: 100%|██████████| 8397/8397 [02:34<00:00, 54.50it/s, saved=475, skipped=575]


----- DONE -----
Total saved: 475
Total skipped: 575


In [ ]:
def make_smooth(cartoon_dir, smooth_dir):
    os.makedirs(smooth_dir, exist_ok=True)

    for fname in tqdm(os.listdir(cartoon_dir)):
        if not fname.endswith(".png"):
            continue

        img = cv2.imread(os.path.join(cartoon_dir, fname))

        # Morphological smoothing — same as original AnimeGAN paper
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (8, 8))
        img_smooth = cv2.erode(img, kernel)
        img_smooth = cv2.GaussianBlur(img_smooth, (3, 3), 0)

        cv2.imwrite(os.path.join(smooth_dir, fname), img_smooth)

make_smooth(
    cartoon_dir="/content/dataset_gan/train/cartoon/style",
    smooth_dir="/content/dataset_gan/train/cartoon/smooth"
)

100%|██████████| 475/475 [00:02<00:00, 184.79it/s]
